# SHIA-RAG — GPU Benchmark (Colab)

Compares **SHIA-RAG** against **RAPTOR, TreeRAG, GraphRAG, HiChunk, and DTE-RAG** on HotpotQA.
All five retrievers are answered by the **same GPU model** (Qwen2.5-3B-Instruct), so any
difference in EM / F1 comes only from the retrieval method. Support-recall measures retrieval directly.

**Step 0 — turn on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save.**
Then run the cells top to bottom. When Cell 2 asks, upload the **shia_core_src.zip** file Claude gave you.


In [ ]:
!pip -q install transformers accelerate scikit-learn datasets pydantic numpy

In [ ]:
# Cell 2 — upload your SHIA-RAG code (shia_core_src.zip)
from google.colab import files
import zipfile, os, sys
print("Choose shia_core_src.zip ...")
up = files.upload()
zname = list(up)[0]
os.makedirs("shia_core", exist_ok=True)
with zipfile.ZipFile(zname) as z: z.extractall("shia_core")
os.makedirs("stubs", exist_ok=True); open("stubs/pymupdf.py","w").write("# stub (PDF path unused)")
sys.path.insert(0, "stubs"); sys.path.insert(0, "shia_core")
print("SHIA-RAG code ready:", os.listdir("shia_core"))

In [ ]:
# Cell 3 — download HotpotQA (distractor dev)
from datasets import load_dataset
N = 50                      # raise to 100+ once it works
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
items = []
for ex in ds.select(range(N)):
    ctx = [[t, s] for t, s in zip(ex["context"]["title"], ex["context"]["sentences"])]
    items.append({"q": ex["question"], "answer": ex["answer"], "context": ctx,
                  "sup": set(ex["supporting_facts"]["title"])})
print(len(items), "questions loaded")

In [ ]:
# Cell 4 — load the GPU reader (same model answers for every method)
import torch, re, string
from transformers import AutoModelForCausalLM, AutoTokenizer
MID = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MID)
model = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16, device_map="auto")
def answer(question, context):
    prompt = ("Answer using ONLY the context. Give the shortest answer possible "
              "(a name, a short phrase, or yes/no). No explanation.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}\nAnswer:")
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=32, do_sample=False)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("reader OK ->", answer("What is the capital of France?", "France's capital is Paris."))

In [ ]:
# Cell 5 — metrics + the five retrievers
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
import numpy as np
from src.pipeline import SHIARAGPipeline
from src.layer0_data_model.schemas import DocumentMimeType

def normalize(s):
    s=(s or "").lower(); s="".join(c for c in s if c not in set(string.punctuation))
    return " ".join(re.sub(r"\b(a|an|the)\b"," ",s).split())
def em(p,g): return float(normalize(p)==normalize(g))
def f1(p,g):
    pt,gt=normalize(p).split(),normalize(g).split()
    if not pt or not gt: return float(pt==gt)
    ns=sum((Counter(pt)&Counter(gt)).values())
    if ns==0: return 0.0
    pr,rc=ns/len(pt),ns/len(gt); return 2*pr*rc/(pr+rc)
def clean_ans(s):
    s=re.sub(r"<think>.*?</think>","",s or "",flags=re.S)
    lines=[l.strip() for l in s.splitlines() if l.strip()]
    s=lines[0] if lines else ""
    return s.strip("\"'.*`: ").strip()

def paragraphs(item):
    return [(t," ".join(x for x in s if x.strip())) for t,s in item["context"] if any(x.strip() for x in s)]

def flat_retrieve(item,budget=400):
    P=paragraphs(item)
    if not P: return "",set()
    v=TfidfVectorizer().fit([p[1] for p in P]+[item["q"]])
    sim=cosine_similarity(v.transform([item["q"]]),v.transform([p[1] for p in P]))[0]
    out,ti,u=[],set(),0
    for i in sim.argsort()[::-1]:
        w=len(P[i][1].split())
        if u+w>budget: break
        out.append(P[i][1]); ti.add(P[i][0]); u+=w
    return "\n".join(out),ti

def raptor_retrieve(item,budget=400):
    P=paragraphs(item)
    if not P: return "",set()
    v=TfidfVectorizer().fit([p[1] for p in P]+[item["q"]])
    Md=v.transform([p[1] for p in P]).toarray(); qv=v.transform([item["q"]]).toarray()[0]
    try:
        km=KMeans(n_clusters=min(max(2,int(len(P)**0.5)),len(P)),n_init=5,random_state=0).fit(Md)
        lab=list(km.labels_); ce=km.cluster_centers_
    except Exception:
        lab=[0]*len(P); ce=np.zeros((1,Md.shape[1]))
    ps=(Md@qv)/((np.linalg.norm(Md,axis=1)*np.linalg.norm(qv))+1e-9)
    cs=(ce@qv)/((np.linalg.norm(ce,axis=1)*np.linalg.norm(qv))+1e-9)
    out,ti,u=[],set(),0
    for i in sorted(range(len(P)),key=lambda i:-(ps[i]+0.3*cs[lab[i]])):
        w=len(P[i][1].split())
        if u+w>budget: break
        out.append(P[i][1]); ti.add(P[i][0]); u+=w
    return "\n".join(out),ti

def treerag_retrieve(item,budget=400):
    P=paragraphs(item)
    if not P: return "",set()
    v=TfidfVectorizer().fit([p[1] for p in P]+[item["q"]])
    sim=cosine_similarity(v.transform([item["q"]]),v.transform([p[1] for p in P]))[0]
    out,ti,u=[],set(),0
    for i in sim.argsort()[::-1]:
        chunk=f"{P[i][0]}: {P[i][1]}"      # ancestor-title prefix
        w=len(chunk.split())
        if u+w>budget: break
        out.append(chunk); ti.add(P[i][0]); u+=w
    return "\n".join(out),ti

def _ents(t): return set(re.findall(r"\b[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*\b",t))
def graphrag_retrieve(item,budget=400):
    P=paragraphs(item)
    if not P: return "",set()
    qe=_ents(item["q"]); pe=[_ents(p[1]) for p in P]; sc=[len(qe&e) for e in pe]
    order=sorted(range(len(P)),key=lambda i:-sc[i]); out,ti,u,pk=[],set(),0,set()
    for i in order:
        if sc[i]==0 and pk: break
        w=len(P[i][1].split())
        if u+w>budget: break
        out.append(P[i][1]); ti.add(P[i][0]); u+=w; pk.add(i)
    if pk:
        seed=set().union(*[pe[i] for i in pk])
        for i in order:
            if i in pk: continue
            if pe[i]&seed:
                w=len(P[i][1].split())
                if u+w>budget: break
                out.append(P[i][1]); ti.add(P[i][0]); u+=w
    return ("\n".join(out),ti) if out else flat_retrieve(item,budget)

def shia_retrieve(item,budget_tokens=2048):
    p=SHIARAGPipeline(); doc=p.register_document("q",DocumentMimeType.PLAIN,pages=1)
    o=1; bt={}
    for pi,(title,sents) in enumerate(item["context"],1):
        for s in sents:
            if s.strip():
                b=p.add_text_block(doc.doc_id,s,o,section_path=f"{pi}.0"); bt[b.block_id]=title; o+=1
    for node,par in p.concept_extractor.extract_from_document(doc,list(p.blocks.values())):
        p.add_concept_node(node,candidate_parents=par)
    p.run_confidence_propagation()
    try: p.run_crosslink_discovery()
    except Exception: pass
    p.validate_invariants()
    rr=p.retrieve(item["q"],token_budget=budget_tokens); seen,txt,ti=set(),[],set()
    for nid in rr.selected_node_ids:
        for bid in getattr(p.nodes[nid],"source_block_ids",[]):
            b=p.blocks.get(bid)
            if b and b.text_content not in seen:
                seen.add(b.text_content); txt.append(b.text_content)
                if bid in bt: ti.add(bt[bid])
    return "\n".join(txt),ti


from collections import defaultdict
def hichunk_retrieve(item,budget=400):
    leaves=[]
    for pi,(title,sents) in enumerate(item["context"]):
        for s in sents:
            if s.strip(): leaves.append((pi,s.strip()))
    if not leaves: return "",set()
    v=TfidfVectorizer().fit([l[1] for l in leaves]+[item["q"]])
    sim=cosine_similarity(v.transform([item["q"]]),v.transform([l[1] for l in leaves]))[0]
    picked=defaultdict(list); u=0
    for j in sim.argsort()[::-1]:
        pi,s=leaves[j]; w=len(s.split())
        if u+w>budget: break
        picked[pi].append(s); u+=w
    out,ti=[],set()
    for pi,ss in picked.items():
        ti.add(item["context"][pi][0])
        if len(ss)>=2: out.append(" ".join(x for x in item["context"][pi][1] if x.strip()))
        else: out.extend(ss)
    return "\n".join(out),ti

def dte_retrieve(item,budget=400):
    P=paragraphs(item)
    if not P: return "",set()
    texts=[p[1] for p in P]
    v=TfidfVectorizer().fit(texts+[item["q"]]); Mp=v.transform(texts)
    sem=cosine_similarity(v.transform([item["q"]]),Mp)[0]
    qt=set(item["q"].lower().split())
    lex=np.array([len(qt&set(t.lower().split()))/((len(qt|set(t.lower().split())))or 1) for t in texts])
    base=0.5*sem+0.5*(lex/(lex.max()+1e-9)); Md=Mp.toarray()
    chosen,ti,u,cand=[],set(),0,list(range(len(P)))
    while cand:
        if not chosen: i=max(cand,key=lambda i:base[i])
        else:
            def mmr(i):
                red=max((Md[i]@Md[j])/((np.linalg.norm(Md[i])*np.linalg.norm(Md[j]))+1e-9) for j in chosen)
                return 0.7*base[i]-0.3*red
            i=max(cand,key=mmr)
        w=len(P[i][1].split())
        if u+w>budget: break
        chosen.append(i); ti.add(P[i][0]); u+=w; cand.remove(i)
    return "\n".join(P[i][1] for i in chosen),ti

RETRIEVERS={"SHIA-RAG":shia_retrieve,"RAPTOR":raptor_retrieve,"TreeRAG":treerag_retrieve,"GraphRAG":graphrag_retrieve,"HiChunk":hichunk_retrieve,"DTE-RAG":dte_retrieve}
print("retrievers ready:", list(RETRIEVERS))

In [ ]:
# Cell 6 — run the benchmark and print the comparison table
import time, json
M=list(RETRIEVERS); agg={m:{"em":[],"f1":[],"rec":[]} for m in M}; rows=[]; t0=time.time()
for i,it in enumerate(items,1):
    row={"q":it["q"],"gold":it["answer"]}
    for m in M:
        ctx,titles=RETRIEVERS[m](it)
        rec=(len(titles & it["sup"])/len(it["sup"])) if it["sup"] else None
        if rec is not None: agg[m]["rec"].append(rec)
        pred=clean_ans(answer(it["q"],ctx))
        agg[m]["em"].append(em(pred,it["answer"])); agg[m]["f1"].append(f1(pred,it["answer"]))
        row[m]=pred
    rows.append(row)
    if i%5==0: print(f"  {i}/{len(items)} done ({time.time()-t0:.0f}s)")
mean=lambda x: round(100*sum(x)/len(x),1) if x else None
print("\n===== RESULTS (%)  ·  reader = Qwen2.5-3B-Instruct (GPU)  ·  n =", len(items), "=====")
print(f"{'Method':<12}{'EM':>8}{'F1':>8}{'SupportRecall':>16}")
for m in M: print(f"{m:<12}{str(mean(agg[m]['em'])):>8}{str(mean(agg[m]['f1'])):>8}{str(mean(agg[m]['rec'])):>16}")
json.dump({"rows":rows}, open("results_gpu.json","w"), indent=2)
print("\nSaved results_gpu.json — same reader for all methods, so differences are purely retrieval.")